# Path Descriptor and Labeling Analysis - Restructured

This notebook performs comprehensive path analysis with symmetric and non-symmetric descriptors.

**Workflow:**
1. Configuration (paths and parameters)
2. Function Definitions (utility functions)
3. Data Processing (symmetric & non-symmetric descriptors)
4. Label Merging & Representative Path Selection
5. Visualization (all plots together)

## 1. Configuration - Paths and Parameters

In [ ]:
# ============================================================================
# CONFIGURATION: Set all paths and parameters here
# ============================================================================
from pathlib import Path
import numpy as np
import pandas as pd

# === DIRECTORY PATHS ===
ROOT_DIR = Path("./example_data")  # TODO: point this at a directory containing path_couplings_timeseries.csv files for one GB structure
OUTPUT_DIR = ROOT_DIR  # Save outputs to same directory

# CSV pattern to find path coupling files
CSV_PATTERN = "*/TRAJ1/path_couplings_timeseries.csv"

# === ANALYSIS PARAMETERS ===
EPS = 1e-12  # Small constant to avoid division by zero

# Representative path selection
N_REPRESENTATIVES_PER_LABEL = 1  # Number of representative paths to select per primary label
FEATURE_COLS_FOR_MEDOID = ["E_dip", "C_dip", "rE", "Lweak_E", "pweak_E", "RE"]

# Merging options for primary labels
INCLUDE_DIFFUSE_AS_5TH_LABEL = True
FLAT_COUNTS_AS_NON_RECOVERING = True

# === OUTPUT FILES ===
OUTPUT_FILES = {
    "symmetric_descriptors": OUTPUT_DIR / "paths_symmetric_descriptors_labeled.csv",
    "order_descriptors": OUTPUT_DIR / "paths_order_descriptors_with_archetype_labels.csv",
    "primary_labels": OUTPUT_DIR / "paths_with_primary_labels.csv",
    "primary_label_counts": OUTPUT_DIR / "primary_label_counts.csv",
    "secondary_cause": OUTPUT_DIR / "secondary_cause_within_primary.csv",
    "secondary_shape": OUTPUT_DIR / "secondary_shape_within_primary.csv",
    "secondary_recovery": OUTPUT_DIR / "secondary_recovery_within_primary.csv",
    "secondary_joint": OUTPUT_DIR / "secondary_joint_cause_shape_recovery_within_primary.csv",
    "representative_paths": OUTPUT_DIR / "representative_paths_for_dynamics.csv",
    "rep_with_symmetric": OUTPUT_DIR / "representative_paths_with_symmetric_labels.csv",
    "all_paths_both_labels": OUTPUT_DIR / "all_paths_with_both_labels.csv",
    "crosstab_full": OUTPUT_DIR / "primary_symmetric_crosstab_full_dataset.csv",
    "crosstab_pct": OUTPUT_DIR / "primary_symmetric_percentage_full_dataset.csv",
}

print("="*70)
print("CONFIGURATION")
print("="*70)
print(f"Root directory: {ROOT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"CSV pattern: {CSV_PATTERN}")
print(f"Representatives per label: {N_REPRESENTATIVES_PER_LABEL}")
print("="*70)

## 2. Function Definitions - Utility Functions

In [ ]:
# ============================================================================
# FILE I/O FUNCTIONS
# ============================================================================

def read_path_csv(csv_path: Path):
    """
    Read path coupling CSV file.
    
    CSV format:
      line1: sites,1276,1834,...
      line2: time,"cpl(1,2)","cpl(2,3)",...
      line3+: numeric values

    Returns
    -------
    sites : list[int]
        Site IDs along the path (length n)
    time : np.ndarray
        Time values, shape (Nt,)
    X : np.ndarray
        Signed couplings, shape (Nt, n-1)
    cpl_cols : list[str]
        Coupling column names
    """
    # Row 1 (sites)
    first = pd.read_csv(csv_path, header=None, nrows=1).iloc[0].tolist()
    if len(first) < 3 or str(first[0]).strip().lower() != "sites":
        raise ValueError(f"{csv_path}: first row must start with 'sites,'")
    
    sites = [int(x) for x in first[1:]]
    
    # Data block with header at row 2
    df = pd.read_csv(csv_path, skiprows=1)
    if "time" not in df.columns:
        raise ValueError(f"{csv_path}: second row must include 'time' column")
    
    time = df["time"].to_numpy(float)
    cpl_cols = [c for c in df.columns if c != "time"]
    X = df[cpl_cols].to_numpy(float)
    
    n = len(sites)
    if X.shape[1] != n - 1:
        raise ValueError(
            f"{csv_path}: sites={n} implies {n-1} dimers, but found {X.shape[1]} coupling columns"
        )
    
    return sites, time, X, cpl_cols

In [ ]:
# ============================================================================
# PER-DIMER METRICS COMPUTATION
# ============================================================================

def per_dimer_vectors_abs(X):
    """
    Compute per-dimer metrics from absolute couplings.
    
    For each dimer i:
      A_i(t) = |T_i(t)|
      J_i = <A_i>  (time-averaged absolute coupling)
      C_i = <A_i>^2 / <A_i^2>  (coherence)
      E_i = J_i * C_i  (effective coupling)
    
    Parameters
    ----------
    X : np.ndarray
        Signed couplings, shape (Nt, m) where m = n-1 dimers
    
    Returns
    -------
    J : np.ndarray
        Mean absolute coupling per dimer, shape (m,)
    C : np.ndarray
        Coherence per dimer, shape (m,)
    A2 : np.ndarray
        Mean squared absolute coupling, shape (m,)
    Astd : np.ndarray
        Std of absolute coupling, shape (m,)  
    E : np.ndarray (optional, for non-symmetric)
        Effective coupling J*C, shape (m,)
    """
    A = np.abs(X)                       # A(t,i) = |T(t,i)|
    J = np.mean(A, axis=0)              # <|T|>
    A2 = np.mean(A * A, axis=0)         # <|T|^2>
    C = (J * J) / (A2 + EPS)            # coherence in [0,1]
    Astd = np.std(A, axis=0, ddof=1) if A.shape[0] > 1 else np.zeros_like(J)
    E = J * C                           # combined effective link
    
    return J, C, A2, Astd, E

In [ ]:
# ============================================================================
# SYMMETRIC DESCRIPTORS (Order-Independent)
# ============================================================================

def summarize_symmetric(J, C):
    """
    Build permutation-invariant descriptors for one path.
    
    Parameters
    ----------
    J : np.ndarray
        Mean absolute coupling per dimer
    C : np.ndarray
        Coherence per dimer
    
    Returns
    -------
    desc : dict
        Dictionary with symmetric descriptor values
    """
    E = J * C
    
    def q(x, p):
        return float(np.percentile(x, p))
    
    desc = {
        # Coupling (absolute mean coupling per dimer) --> 7 feature descriptors
        "J_mean": float(np.mean(J)),
        "J_std": float(np.std(J, ddof=1)) if len(J) > 1 else 0.0,
        "J_min": float(np.min(J)),
        "J_med": q(J, 50),
        "J_q10": q(J, 10),
        "J_q90": q(J, 90),
        "J_min_over_med": float(np.min(J) / (q(J, 50) + EPS)),
        
        # Coherence --> 7 feature descriptors
        "C_mean": float(np.mean(C)),
        "C_std": float(np.std(C, ddof=1)) if len(C) > 1 else 0.0,
        "C_min": float(np.min(C)),
        "C_med": q(C, 50),
        "C_q10": q(C, 10),
        "C_q90": q(C, 90),
        "frac_C_lt_0p5": float(np.mean(C < 0.5)),  # fraction with coherence < 0.4
        
        # Combined effective (coupling weighted by coherence) --> 5 feature descriptors
        "E_mean": float(np.mean(E)),
        "E_min": float(np.min(E)),
        "E_med": q(E, 50),
        "E_q10": q(E, 10),
        "E_min_over_med": float(np.min(E) / (q(E, 50) + EPS)),
    }
    return desc


def label_from_symmetric(desc_df: pd.DataFrame):
    """
    Assign symmetric labels based on overall path characteristics.
    
    Labels: strong/weak-coherent/fluctuating[-heterogeneous]
    """
    df = desc_df.copy()
    
    # Global thresholds (data-driven)
    Jm_med = float(np.percentile(df["J_mean"], 50))
    Cm_med = float(np.percentile(df["C_mean"], 50))
    Jstd_q75 = float(np.percentile(df["J_std"], 75))
    Jminratio_q10 = float(np.percentile(df["J_min_over_med"], 10))
    
    labels = []
    for _, r in df.iterrows():
        strength = "strong" if r["J_mean"] >= Jm_med else "weak"
        stability = "coherent" if r["C_mean"] >= Cm_med else "fluctuating"
        hetero = (r["J_std"] >= Jstd_q75) or (r["J_min_over_med"] <= Jminratio_q10)
        
        if hetero:
            labels.append(f"{strength}-{stability}-heterogeneous")
        else:
            labels.append(f"{strength}-{stability}")
    
    df["label"] = labels
    df.attrs["thresholds"] = {
        "J_mean_median": Jm_med,
        "C_mean_median": Cm_med,
        "J_std_q75": Jstd_q75,
        "J_min_over_med_q10": Jminratio_q10,
    }
    return df

In [ ]:
# ============================================================================
# NON-SYMMETRIC DESCRIPTORS (Order-Aware) - Helper Functions
# ============================================================================

def longest_run_true(b):
    """Longest consecutive run of True in a boolean 1D array."""
    best = cur = 0
    for x in b:
        if bool(x):
            cur += 1
            best = max(best, cur)
        else:
            cur = 0
    return best


def biggest_drop_rise(x):
    """
    Find positions and magnitudes of biggest drop and rise.
    
    Returns
    -------
    p_drop : float
        Normalized position (0,1] where biggest drop occurs
    drop_mag : float
        Magnitude of biggest drop
    p_rise : float
        Normalized position (0,1] where biggest rise occurs
    rise_mag : float
        Magnitude of biggest rise
    """
    m = len(x)
    if m < 3:
        return np.nan, np.nan, np.nan, np.nan
    
    dx = np.diff(x)
    i_drop = int(np.argmin(dx)) + 1
    i_rise = int(np.argmax(dx)) + 1
    
    p_drop = i_drop / (m - 1)
    p_rise = i_rise / (m - 1)
    drop_mag = float(-np.min(dx))
    rise_mag = float(np.max(dx))
    
    return p_drop, drop_mag, p_rise, rise_mag


def end_mid_end_dip_score(x):
    """
    Detect 'dip in the middle with recovery at both ends' using quarters.
    
    Returns
    -------
    end_mean : float
        Mean of first and last quarters
    mid_mean : float
        Mean of middle half
    dip_frac : float
        (end_mean - mid_mean) / end_mean
        Positive = middle is weaker (GB-like dip)
    """
    m = len(x)
    q = max(1, int(0.25 * m))
    first = x[:q]
    last = x[-q:]
    mid = x[q:-q] if (m - 2 * q) >= 1 else x
    
    end_mean = 0.5 * (np.mean(first) + np.mean(last))
    mid_mean = np.mean(mid)
    dip_frac = (end_mean - mid_mean) / (end_mean + EPS)
    
    return float(end_mean), float(mid_mean), float(dip_frac)

In [ ]:
# ============================================================================
# NON-SYMMETRIC DESCRIPTORS (Order-Aware) - Main Function
# ============================================================================

def order_descriptors_full(J, C, E):
    """
    Produces order-aware descriptors that respect source->target ordering.
    
    Captures:
    - Bottleneck positions (pJ, pC, pE)
    - Bottleneck depths (rJ, rC, rE)  
    - Mid-dip relative to ends (J_dip, C_dip, E_dip)
    - Barrier and recovery (drop_*, rise_*)
    - Weak region width and center (Lweak_*, pweak_*)
    - Recovery behavior from early to late (RJ, RC, RE)
    """
    m = len(J)
    
    # Bottleneck positions (normalized)
    pJ = (int(np.argmin(J)) + 1) / m
    pC = (int(np.argmin(C)) + 1) / m
    pE = (int(np.argmin(E)) + 1) / m
    
    # Bottleneck depth ratios vs typical
    rJ = float(np.min(J) / (np.median(J) + EPS))
    rC = float(np.min(C) / (np.median(C) + EPS))
    rE = float(np.min(E) / (np.median(E) + EPS))
    
    # Early vs late (halves)
    mid = m // 2
    Je, Jl = np.mean(J[:mid]), np.mean(J[mid:])
    Ce, Cl = np.mean(C[:mid]), np.mean(C[mid:])
    Ee, El = np.mean(E[:mid]), np.mean(E[mid:])
    
    RJ = float(Jl / (Je + EPS))
    RC = float(Cl / (Ce + EPS))
    RE = float(El / (Ee + EPS))
    
    # End-mid-end dip signature (GB dip + recovery)
    J_end, J_mid, J_dip = end_mid_end_dip_score(J)
    C_end, C_mid, C_dip = end_mid_end_dip_score(C)
    E_end, E_mid, E_dip = end_mid_end_dip_score(E)
    
    # Drop/rise analysis
    pdJ, dJ, prJ, rJ_val = biggest_drop_rise(J)
    pdC, dC, prC, rC_val = biggest_drop_rise(C)
    pdE, dE, prE, rE_val = biggest_drop_rise(E)
    
    # Weak region analysis
    J_thr = 0.5 * (J.min() + J.max())
    C_thr = 0.5 * (C.min() + C.max())
    E_thr = 0.5 * (E.min() + E.max())
    
    Lweak_J = longest_run_true(J < J_thr)
    Lweak_C = longest_run_true(C < C_thr)
    Lweak_E = longest_run_true(E < E_thr)
    
    # Weak region center (rough)
    pweak_J = (np.argmin(J) + 1) / m
    pweak_C = (np.argmin(C) + 1) / m
    pweak_E = (np.argmin(E) + 1) / m
    
    return {
        # Bottleneck positions
        "pJ": pJ, "pC": pC, "pE": pE,
        # Bottleneck depths
        "rJ": rJ, "rC": rC, "rE": rE,
        # Recovery ratios
        "RJ": RJ, "RC": RC, "RE": RE,
        # Dip scores
        "J_dip": J_dip, "C_dip": C_dip, "E_dip": E_dip,
        # Drop/rise
        "drop_J": dJ, "rise_J": rJ_val, "pdrop_J": pdJ, "prise_J": prJ,
        "drop_C": dC, "rise_C": rC_val, "pdrop_C": pdC, "prise_C": prC,
        "drop_E": dE, "rise_E": rE_val, "pdrop_E": pdE, "prise_E": prE,
        # Weak region
        "Lweak_J": Lweak_J, "Lweak_C": Lweak_C, "Lweak_E": Lweak_E,
        "pweak_J": pweak_J, "pweak_C": pweak_C, "pweak_E": pweak_E,
    }


def label_archetypes(desc_df: pd.DataFrame):
    """
    Assign archetype labels based on order-aware descriptors.
    
    Creates 4 label columns:
    - label_location: GB-limited / source-limited / target-limited
    - label_cause: coupling-limited / coherence-limited / mixed-limited
    - label_shape: sharp / extended / diffuse / sharp+extended
    - label_recovery: recovering / degrading / flat
    """
    out = desc_df.copy()
    
    # Define thresholds using quantiles
    pE_lo = float(np.percentile(out["pE"], 20))
    pE_hi = float(np.percentile(out["pE"], 80))
    
    Edip_q75 = float(np.percentile(out["E_dip"], 75))
    Cdip_q75 = float(np.percentile(out["C_dip"], 75))
    
    rE_q25 = float(np.percentile(out["rE"], 25))
    rJ_q25 = float(np.percentile(out["rJ"], 25))
    rC_q25 = float(np.percentile(out["rC"], 25))
    
    LwE_q75 = float(np.percentile(out["Lweak_E"], 75))
    LwC_q75 = float(np.percentile(out["Lweak_C"], 75))
    
    dropE_q75 = float(np.percentile(out["drop_E"], 75))
    riseE_q75 = float(np.percentile(out["rise_E"], 75))
    dropC_q75 = float(np.percentile(out["drop_C"], 75))
    riseC_q75 = float(np.percentile(out["rise_C"], 75))
    
    rec_hi = 1.05
    rec_lo = 0.95
    
    thresholds = {
        "pE_lo_q20": pE_lo, "pE_hi_q80": pE_hi,
        "E_dip_q75": Edip_q75, "C_dip_q75": Cdip_q75,
        "rE_q25": rE_q25, "rJ_q25": rJ_q25, "rC_q25": rC_q25,
        "Lweak_E_q75": LwE_q75, "Lweak_C_q75": LwC_q75,
        "drop_E_q75": dropE_q75, "rise_E_q75": riseE_q75,
        "drop_C_q75": dropC_q75, "rise_C_q75": riseC_q75,
        "recovery_hi": rec_hi, "recovery_lo": rec_lo,
    }
    
    loc_labels, cause_labels, shape_labels, rec_labels, combined = [], [], [], [], []
    
    for _, r in out.iterrows():
        # Location
        if r["pE"] <= pE_lo:
            label_location = "source-limited"
        elif r["pE"] >= pE_hi:
            label_location = "target-limited"
        else:
            label_location = "GB-limited"
        
        # Cause
        coupling_signal = ((r["E_dip"] >= Edip_q75) or (r["drop_E"] >= dropE_q75) or 
                          (r["Lweak_E"] >= LwE_q75) or (r["rJ"] <= rJ_q25))
        coherence_signal = ((r["C_dip"] >= Cdip_q75) or (r["drop_C"] >= dropC_q75) or 
                           (r["Lweak_C"] >= LwC_q75) or (r["rC"] <= rC_q25))
        
        if coupling_signal and not coherence_signal:
            label_cause = "coupling-limited"
        elif coherence_signal and not coupling_signal:
            label_cause = "coherence-limited"
        else:
            label_cause = "mixed-limited"
        
        # Shape
        sharp = ((r["drop_E"] >= dropE_q75) and (r["rise_E"] >= riseE_q75)) or \
                ((r["drop_C"] >= dropC_q75) and (r["rise_C"] >= riseC_q75))
        extended = (r["Lweak_E"] >= LwE_q75) or (r["Lweak_C"] >= LwC_q75)
        
        if sharp and extended:
            label_shape = "sharp+extended"
        elif sharp:
            label_shape = "sharp"
        elif extended:
            label_shape = "extended"
        else:
            label_shape = "diffuse"
        
        # Recovery
        if r["RE"] > rec_hi:
            label_recovery = "recovering"
        elif r["RE"] < rec_lo:
            label_recovery = "degrading"
        else:
            label_recovery = "flat"
        
        # Combined
        label_combined = f"{label_location}, {label_cause}, {label_shape}, {label_recovery}"
        
        loc_labels.append(label_location)
        cause_labels.append(label_cause)
        shape_labels.append(label_shape)
        rec_labels.append(label_recovery)
        combined.append(label_combined)
    
    out["label_location"] = loc_labels
    out["label_cause"] = cause_labels
    out["label_shape"] = shape_labels
    out["label_recovery"] = rec_labels
    out["label_combined"] = combined
    out.attrs["thresholds"] = thresholds
    
    return out

In [ ]:
# ============================================================================
# LABEL MERGING FUNCTIONS
# ============================================================================

def add_primary_label(
    df: pd.DataFrame,
    include_diffuse_as_5th: bool = True,
    flat_counts_as_nonrecovering: bool = True,
):
    """
    Creates 'label_primary' column with 4-5 primary classification labels.
    
    Primary labels:
    1) GB-crossing (recovering)
    2) GB-blocked (non-recovering)
    3) Source-limited
    4) Target-limited
    5) Diffuse/weakly structured (optional)
    """
    out = df.copy()
    
    required = ["label_location", "label_recovery", "label_shape"]
    missing = [c for c in required if c not in out.columns]
    if missing:
        raise ValueError(f"Missing columns for merging primary labels: {missing}")
    
    # Define non-recovering condition
    if flat_counts_as_nonrecovering:
        nonrec = out["label_recovery"].isin(["degrading", "flat"])
    else:
        nonrec = out["label_recovery"].eq("degrading")
    
    # Initialize
    out["label_primary"] = "Unassigned"
    
    # Optional: diffuse as separate class
    if include_diffuse_as_5th:
        out.loc[out["label_shape"].eq("diffuse"), "label_primary"] = "Diffuse/weakly structured"
    
    # Source/Target limited (geometry dominant)
    out.loc[out["label_location"].eq("source-limited"), "label_primary"] = "Source-limited"
    out.loc[out["label_location"].eq("target-limited"), "label_primary"] = "Target-limited"
    
    # GB-limited split by recovery
    gb = out["label_location"].eq("GB-limited")
    out.loc[gb & out["label_recovery"].eq("recovering"), "label_primary"] = "GB-crossing (recovering)"
    out.loc[gb & nonrec, "label_primary"] = "GB-blocked (non-recovering)"
    
    return out


def secondary_composition_tables(df: pd.DataFrame, primary_col="label_primary"):
    """
    Generate composition tables showing secondary label distributions within primary labels.
    
    Returns
    -------
    summary_primary : DataFrame
        Counts and percentages for each primary label
    comp_cause : DataFrame
        Distribution of label_cause within each primary label
    comp_shape : DataFrame
        Distribution of label_shape within each primary label
    comp_recovery : DataFrame
        Distribution of label_recovery within each primary label
    comp_joint : DataFrame
        Distribution of (cause, shape, recovery) combos within each primary label
    """
    if primary_col not in df.columns:
        raise ValueError(f"'{primary_col}' not found. Run add_primary_label() first.")
    
    # Primary counts
    summary_primary = (
        df[primary_col]
        .value_counts(dropna=False)
        .rename_axis(primary_col)
        .reset_index(name="count")
    )
    summary_primary["percent"] = 100.0 * summary_primary["count"] / summary_primary["count"].sum()
    
    # Helper to build within-primary distribution
    def within_dist(col):
        if col not in df.columns:
            return None
        tmp = (
            df.groupby([primary_col, col], dropna=False)
              .size()
              .reset_index(name="count")
        )
        tmp["percent_within_primary"] = tmp["count"] / tmp.groupby(primary_col)["count"].transform("sum") * 100.0
        tmp = tmp.sort_values([primary_col, "count"], ascending=[True, False])
        return tmp
    
    comp_cause = within_dist("label_cause")
    comp_shape = within_dist("label_shape")
    comp_recovery = within_dist("label_recovery")
    
    # Joint secondary signature table
    cols_joint = [c for c in ["label_cause", "label_shape", "label_recovery"] if c in df.columns]
    if len(cols_joint) == 3:
        comp_joint = (
            df.groupby([primary_col] + cols_joint, dropna=False)
              .size()
              .reset_index(name="count")
        )
        comp_joint["percent_within_primary"] = comp_joint["count"] / comp_joint.groupby(primary_col)["count"].transform("sum") * 100.0
        comp_joint = comp_joint.sort_values([primary_col, "count"], ascending=[True, False])
    else:
        comp_joint = None
    
    return summary_primary, comp_cause, comp_shape, comp_recovery, comp_joint

In [ ]:
# ============================================================================
# REPRESENTATIVE PATH SELECTION
# ============================================================================

from sklearn.preprocessing import StandardScaler

def select_representative_paths(
    df: pd.DataFrame,
    primary_col: str = "label_primary",
    feature_cols=None,
    n_per_label: int = 1,
    within_class_standardize: bool = True,
):
    """
    Select representative paths (medoids) closest to class centroids.
    
    Parameters
    ----------
    df : DataFrame
        Dataset with primary labels and feature columns
    primary_col : str
        Column holding primary labels
    feature_cols : list[str] or None
        Descriptor columns for representativeness (uses defaults if None)
    n_per_label : int
        Number of paths per primary label
    within_class_standardize : bool
        Standardize features within each class before computing centroid
    
    Returns
    -------
    rep_df : DataFrame
        Representative paths with 'rep_rank' and 'rep_dist' columns
    """
    if feature_cols is None:
        feature_cols = FEATURE_COLS_FOR_MEDOID
    
    missing = [c for c in feature_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing feature columns: {missing}")
    if primary_col not in df.columns:
        raise ValueError(f"Missing primary label column: {primary_col}")
    if n_per_label < 1:
        raise ValueError("n_per_label must be >= 1")
    
    reps = []
    
    for label, sub in df.groupby(primary_col, sort=True):
        sub = sub.copy()
        
        # Drop rows with NaNs in feature space
        good = sub[feature_cols].notna().all(axis=1)
        sub_good = sub.loc[good].copy()
        
        if len(sub_good) == 0:
            # Fallback: take first n_per_label rows
            take = sub.head(min(n_per_label, len(sub))).copy()
            take["rep_dist"] = np.nan
            take["rep_rank"] = np.arange(1, len(take) + 1)
            reps.append(take)
            continue
        
        X = sub_good[feature_cols].to_numpy(float)
        
        if within_class_standardize and len(sub_good) > 1:
            scaler = StandardScaler()
            Xs = scaler.fit_transform(X)
        else:
            Xs = X
        
        centroid = Xs.mean(axis=0)
        dists = np.linalg.norm(Xs - centroid, axis=1)
        sub_good["rep_dist"] = dists
        
        sub_good = sub_good.sort_values("rep_dist", ascending=True)
        k = min(n_per_label, len(sub_good))
        chosen = sub_good.head(k).copy()
        chosen["rep_rank"] = np.arange(1, k + 1)
        
        reps.append(chosen)
    
    rep_df = pd.concat(reps, ignore_index=True)
    return rep_df

## 3. Data Processing Workflow

In [ ]:
# ============================================================================
# STEP 1: Build Symmetric Descriptors Dataset
# ============================================================================

def build_symmetric_dataset(root_dir: Path, pattern: str, save_vectors=False):
    """
    Process all paths and compute symmetric (order-independent) descriptors.
    """
    csv_files = sorted(root_dir.glob(pattern))
    if not csv_files:
        raise ValueError(f"No CSV files found under {root_dir} matching pattern {pattern}")
    
    rows = []
    for csv_path in csv_files:
        sites, time, X, cpl_cols = read_path_csv(csv_path)
        J, C, A2, Astd, E = per_dimer_vectors_abs(X)
        desc = summarize_symmetric(J, C)
        
        desc["path"] = str(csv_path)
        desc["n_sites"] = len(sites)
        desc["n_dimers"] = len(J)
        
        rows.append(desc)
        
        if save_vectors:
            # Save per-dimer vectors
            out = pd.DataFrame({
                "dimer_i": np.arange(1, len(J)+1),
                "site_i": sites[:-1],
                "site_j": sites[1:],
                "J_abs_mean": J,
                "C_coherence": C,
                "E_JxC": E,
                "abs_std": Astd,
                "abs_mean_sq": A2
            })
            out.to_csv(csv_path.with_suffix(".J_C_vectors.csv"), index=False)
    
    return pd.DataFrame(rows)


print("Processing symmetric descriptors...")
desc_symmetric = build_symmetric_dataset(ROOT_DIR, CSV_PATTERN, save_vectors=True)
labeled_symmetric = label_from_symmetric(desc_symmetric)

# Save
labeled_symmetric.to_csv(OUTPUT_FILES["symmetric_descriptors"], index=False)

print(f"\nSymmetric descriptors saved to: {OUTPUT_FILES['symmetric_descriptors']}")
print("Thresholds used:", labeled_symmetric.attrs["thresholds"])
print("\nSymmetric label distribution:")
print(labeled_symmetric["label"].value_counts())

In [ ]:
# ============================================================================
# STEP 2: Build Non-Symmetric (Order-Aware) Descriptors Dataset
# ============================================================================

def build_order_dataset(root_dir: Path, pattern: str, save_vectors=False):
    """
    Process all paths and compute order-aware descriptors.
    """
    csv_files = sorted(root_dir.glob(pattern))
    if not csv_files:
        raise ValueError(f"No CSV files found under {root_dir} matching pattern {pattern}")
    
    rows = []
    for csv_path in csv_files:
        sites, time, X, cpl_cols = read_path_csv(csv_path)
        J, C, A2, Astd, E = per_dimer_vectors_abs(X)
        
        desc = order_descriptors_full(J, C, E)
        desc["path"] = str(csv_path)
        desc["n_sites"] = len(sites)
        desc["n_dimers"] = len(J)
        
        rows.append(desc)
        
        if save_vectors:
            # Save JCE vectors
            vec = pd.DataFrame({
                "dimer_i": np.arange(1, len(J)+1),
                "site_i": sites[:-1],
                "site_j": sites[1:],
                "J_abs_mean": J,
                "C_coherence": C,
                "E_JxC": E,
            })
            vec.to_csv(csv_path.with_suffix(".JCE_vectors.csv"), index=False)
    
    return pd.DataFrame(rows)


print("Processing order-aware descriptors...")
desc_order = build_order_dataset(ROOT_DIR, CSV_PATTERN, save_vectors=True)
labeled_order = label_archetypes(desc_order)

# Save
labeled_order.to_csv(OUTPUT_FILES["order_descriptors"], index=False)

print(f"\nOrder descriptors saved to: {OUTPUT_FILES['order_descriptors']}")
print("Thresholds used:", labeled_order.attrs.get("thresholds", {}))
print("\nTop combined labels:")
print(labeled_order["label_combined"].value_counts().head(25))

In [ ]:
# ============================================================================
# STEP 3: Merge into Primary Labels
# ============================================================================

print("Merging into primary labels...")
df_primary = add_primary_label(
    labeled_order,
    include_diffuse_as_5th=INCLUDE_DIFFUSE_AS_5TH_LABEL,
    flat_counts_as_nonrecovering=FLAT_COUNTS_AS_NON_RECOVERING
)

# Generate secondary composition tables
summary_primary, comp_cause, comp_shape, comp_recovery, comp_joint = secondary_composition_tables(df_primary)

# Save all outputs
df_primary.to_csv(OUTPUT_FILES["primary_labels"], index=False)
summary_primary.to_csv(OUTPUT_FILES["primary_label_counts"], index=False)

if comp_cause is not None:
    comp_cause.to_csv(OUTPUT_FILES["secondary_cause"], index=False)
if comp_shape is not None:
    comp_shape.to_csv(OUTPUT_FILES["secondary_shape"], index=False)
if comp_recovery is not None:
    comp_recovery.to_csv(OUTPUT_FILES["secondary_recovery"], index=False)
if comp_joint is not None:
    comp_joint.to_csv(OUTPUT_FILES["secondary_joint"], index=False)

print(f"\nPrimary labels saved to: {OUTPUT_FILES['primary_labels']}")
print("\nPrimary label distribution:")
print(summary_primary)

In [ ]:
# ============================================================================
# STEP 4: Select Representative Paths
# ============================================================================

print("Selecting representative paths (medoids)...")
rep_paths = select_representative_paths(
    df_primary,
    n_per_label=N_REPRESENTATIVES_PER_LABEL,
    feature_cols=FEATURE_COLS_FOR_MEDOID
)

# Save
rep_paths.to_csv(OUTPUT_FILES["representative_paths"], index=False)

print(f"\nRepresentative paths saved to: {OUTPUT_FILES['representative_paths']}")
print(f"\nSelected {len(rep_paths)} representative paths:")
print(rep_paths[["label_primary", "rep_rank", "rep_dist", "path"]])

In [ ]:
# ============================================================================
# STEP 5: Add Symmetric Labels to Representative Paths
# ============================================================================

print("Adding symmetric labels to representative paths...")
rep_with_sym = rep_paths.merge(
    labeled_symmetric[['path', 'label']],
    on='path',
    how='left'
)
rep_with_sym.rename(columns={'label': 'label_symmetric'}, inplace=True)

# Save
rep_with_sym.to_csv(OUTPUT_FILES["rep_with_symmetric"], index=False)

print(f"\nRepresentative paths with symmetric labels saved to: {OUTPUT_FILES['rep_with_symmetric']}")
print("\nMapping of primary → symmetric labels for representatives:")
print(rep_with_sym[['label_primary', 'label_symmetric', 'path']])

In [ ]:
# ============================================================================
# STEP 6: Comprehensive Analysis (All Paths with Both Labels)
# ============================================================================

print("Generating comprehensive analysis...")
comprehensive = df_primary.merge(
    labeled_symmetric[['path', 'label']],
    on='path',
    how='left'
)
comprehensive.rename(columns={'label': 'label_symmetric'}, inplace=True)

# Save outputs
comprehensive.to_csv(OUTPUT_FILES["all_paths_both_labels"], index=False)

# Cross-tabulation
crosstab = pd.crosstab(
    comprehensive['label_primary'],
    comprehensive['label_symmetric'],
    margins=True
)
crosstab.to_csv(OUTPUT_FILES["crosstab_full"])

# Percentage table
crosstab_pct = pd.crosstab(
    comprehensive['label_primary'],
    comprehensive['label_symmetric'],
    normalize='index'
) * 100
crosstab_pct.to_csv(OUTPUT_FILES["crosstab_pct"])

print(f"\nComprehensive analysis complete!")
print(f"  All paths with both labels: {OUTPUT_FILES['all_paths_both_labels']}")
print(f"  Cross-tabulation: {OUTPUT_FILES['crosstab_full']}")
print(f"  Percentage table: {OUTPUT_FILES['crosstab_pct']}")

print("\n" + "="*70)
print("DATA PROCESSING COMPLETE")
print("="*70)
print(f"\nTotal paths analyzed: {len(comprehensive)}")
print("\nPrimary label distribution:")
print(comprehensive['label_primary'].value_counts())
print("\nSymmetric label distribution:")
print(comprehensive['label_symmetric'].value_counts())

## 4. Visualization - All Plots Together

In [ ]:
# ============================================================================
# PLOTTING SETUP
# ============================================================================

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Set plotting style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# ============================================================================
# PLOT 1: PCA Medoid Selection Visualization
# ============================================================================

## Load datasets for plotting
df_primary = pd.read_csv(OUTPUT_FILES["primary_labels"])
rep_paths = pd.read_csv(OUTPUT_FILES["representative_paths"])
comprehensive = pd.read_csv(OUTPUT_FILES["all_paths_both_labels"])
labeled_symmetric = pd.read_csv(OUTPUT_FILES["symmetric_descriptors"])

def plot_medoid_selection_for_label(
    df: pd.DataFrame,
    label_value: str,
    primary_col: str = "label_primary",
    feature_cols=None,
    n_select: int = 3,
    within_class_standardize: bool = True,
    annotate: bool = True,
 ):
    """
    Visualize how representative paths (medoids) are selected for ONE primary label.
    Shows 2D PCA projection with centroid and selected paths.
    """
    if feature_cols is None:
        feature_cols = FEATURE_COLS_FOR_MEDOID
    
    # Filter to one label
    sub = df[df[primary_col] == label_value].copy()
    if len(sub) == 0:
        raise ValueError(f"No rows for {primary_col} == '{label_value}'")
    
    # Keep only complete rows
    sub = sub.dropna(subset=feature_cols).copy()
    if len(sub) < 2:
        raise ValueError(f"Not enough complete rows in '{label_value}' (need >=2)")
    
    X = sub[feature_cols].to_numpy(float)
    
    # Standardize within class
    if within_class_standardize:
        scaler = StandardScaler()
        Xs = scaler.fit_transform(X)
    else:
        Xs = X
    
    # Centroid
    centroid = Xs.mean(axis=0)
    dists = np.linalg.norm(Xs - centroid, axis=1)
    order = np.argsort(dists)
    k = min(n_select, len(sub))
    chosen_idx = order[:k]
    
    # PCA to 2D
    pca = PCA(n_components=2)
    X2 = pca.fit_transform(Xs)
    c2 = pca.transform(centroid.reshape(1, -1))[0]
    
    # Print PCA interpretation
    print(f"\n{'='*60}")
    print(f"PCA Interpretation for '{label_value}'")
    print(f"{'='*60}")
    print(f"\nExplained variance:")
    print(f"  PC1: {pca.explained_variance_ratio_[0]:.1%}")
    print(f"  PC2: {pca.explained_variance_ratio_[1]:.1%}")
    print(f"  Total: {pca.explained_variance_ratio_.sum():.1%}")
    
    loadings = pd.DataFrame(
        pca.components_.T,
        columns=['PC1', 'PC2'],
        index=feature_cols
    )
    print(f"\nFeature loadings:")
    print(loadings.round(3))
    print(f"{'='*60}\n")
    
    # Plot
    plt.figure(figsize=(9, 6))
    plt.scatter(X2[:, 0], X2[:, 1], alpha=0.7, marker="o", label="Paths")
    plt.scatter([c2[0]], [c2[1]], marker="*", s=200, color="red", label="Centroid")
    plt.scatter(
        X2[chosen_idx, 0], X2[chosen_idx, 1],
        s=140, marker="o", edgecolors='black', linewidths=2, label="Selected medoids"
    )
    
    # Lines from chosen to centroid
    for rank, idx_i in enumerate(chosen_idx, start=1):
        plt.plot([c2[0], X2[idx_i, 0]], [c2[1], X2[idx_i, 1]], 'k--', linewidth=1, alpha=0.5)
        if annotate:
            plt.text(X2[idx_i, 0], X2[idx_i, 1], f"#{rank}", fontsize=10, fontweight='bold')
    
    plt.title(f"Medoid Selection: '{label_value}'\\n(PCA projection of feature space)")
    plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
    plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print selected paths
    chosen = sub.iloc[chosen_idx].copy()
    chosen["rep_rank"] = np.arange(1, k + 1)
    chosen["rep_dist"] = dists[chosen_idx]
    chosen["subdir"] = chosen["path"].apply(lambda p: Path(p).parent.parent.name)
    chosen["sites"] = chosen["path"].apply(lambda p: " ".join(map(str, read_path_csv(Path(p))[0])))
    
    print("\nSelected representative paths:")
    with pd.option_context(
        "display.max_rows", None,
        "display.max_columns", None,
        "display.max_colwidth", None,
        "display.width", 200,
    ):
        print(chosen[[primary_col, "rep_rank", "rep_dist", "subdir", "sites"]].to_string(index=False))
    
    return chosen


 # Example: Plot one primary label
if len(df_primary) > 0:
    available_labels = df_primary['label_primary'].unique()
    print(f"\nAvailable primary labels: {list(available_labels)}")
    
    # Plot first available label (change as needed)
    label_to_plot = available_labels[3]
    print(f"\nPlotting medoid selection for: {label_to_plot}")
    
    plot_medoid_selection_for_label(
        df_primary,
        label_value=label_to_plot,
        n_select=min(10, 10),
        annotate=True
    )

In [ ]:
# ============================================================================
# PLOT 4: Per-Path Coupling Profiles (Representative Paths)
# ============================================================================

def plot_representative_path_profiles(rep_df: pd.DataFrame, max_plots=4):
    """
    Plot J, C, E profiles for representative paths from each primary label.
    """
    primary_labels = rep_df['label_primary'].unique()
    n_labels = min(len(primary_labels), max_plots)
    
    fig, axes = plt.subplots(n_labels, 3, figsize=(14, 4*n_labels))
    if n_labels == 1:
        axes = axes.reshape(1, -1)
    
    for i, label in enumerate(primary_labels[:n_labels]):
        rows = rep_df[rep_df['label_primary'] == label]
        if len(rows) == 0:
            continue
        
        # Take first representative
        row = rows.iloc[0]
        csv_path = Path(row['path'])
        
        # Read data
        sites, time, X, _ = read_path_csv(csv_path)
        J, C, A2, Astd, E = per_dimer_vectors_abs(X)
        
        dimer_idx = np.arange(1, len(J) + 1)
        
        # Plot J
        axes[i, 0].plot(dimer_idx, J, 'o-', color='steelblue')
        axes[i, 0].set_ylabel('J (time-avg |coupling|)')
        axes[i, 0].set_title(f'{label}\\nCoupling Profile')
        axes[i, 0].grid(alpha=0.3)
        
        # Plot C
        axes[i, 1].plot(dimer_idx, C, 'o-', color='coral')
        axes[i, 1].set_ylabel('C (coherence)')
        axes[i, 1].set_title('Coherence Profile')
        axes[i, 1].grid(alpha=0.3)
        axes[i, 1].axhline(0.5, color='gray', linestyle='--', alpha=0.5, label='C=0.5')
        axes[i, 1].legend()
        
        # Plot E
        axes[i, 2].plot(dimer_idx, E, 'o-', color='green')
        axes[i, 2].set_ylabel('E (J × C)')
        axes[i, 2].set_title('Effective Coupling Profile')
        axes[i, 2].grid(alpha=0.3)
        
        # X-label only on bottom row
        if i == n_labels - 1:
            for ax in axes[i]:
                ax.set_xlabel('Dimer position along path')
    
    plt.tight_layout()
    plt.show()


if len(rep_paths) > 0:
    print("\nGenerating representative path profiles...")
    plot_representative_path_profiles(rep_paths, max_plots=4)

In [ ]:
# ============================================================================
# PLOT 5: Descriptor Distributions by Primary Label
# ============================================================================

def plot_descriptor_distributions(df: pd.DataFrame, descriptors=['E_dip', 'C_dip', 'rE', 'RE']):
    """
    Violin plots showing distribution of key descriptors across primary labels.
    """
    available_desc = [d for d in descriptors if d in df.columns]
    if not available_desc:
        print("No descriptor columns found for plotting")
        return
    
    n_desc = len(available_desc)
    fig, axes = plt.subplots(1, n_desc, figsize=(5*n_desc, 5))
    if n_desc == 1:
        axes = [axes]
    
    for i, desc in enumerate(available_desc):
        # Prepare data
        plot_data = []
        labels_list = []
        for label in df['label_primary'].unique():
            values = df[df['label_primary'] == label][desc].dropna()
            plot_data.append(values)
            labels_list.append(label)
        
        # Violin plot
        parts = axes[i].violinplot(plot_data, positions=range(len(labels_list)),
                                     showmeans=True, showmedians=True)
        
        axes[i].set_xticks(range(len(labels_list)))
        axes[i].set_xticklabels(labels_list, rotation=45, ha='right')
        axes[i].set_ylabel(desc)
        axes[i].set_title(f'{desc} Distribution')
        axes[i].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()


if len(df_primary) > 0:
    print("\nGenerating descriptor distribution plots...")
    plot_descriptor_distributions(df_primary)

In [ ]:
## 3D visualization of representative paths with bottleneck highlighting

# ============================================================================
# This cell can be run independently after completing the data processing steps
# 
# Required files:
#   1. representative_paths_for_dynamics.csv (OUTPUT_FILES["representative_paths"])
#   2. averaged_ham.gro (geometry file with molecular coordinates)
#   3. *.JCE_vectors.csv files (generated during order descriptor processing)
#
# Configuration:
#   - Update ROOT_DIR and OUTPUT_DIR in Configuration section (Cell 3)
#   - Update GRO_FILE path below if needed
# ============================================================================

import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import MDAnalysis as mda
from pathlib import Path
import numpy as np
import pandas as pd

def read_gro_file(gro_path):
    """
    Read a .gro file using MDAnalysis and extract center-of-mass coordinates for each residue.
    
    Returns:
        dict: {residue_number: (x, y, z)} coordinates in nm
    """
    # Load structure with MDAnalysis
    u = mda.Universe(str(gro_path))
    
    com_coords = {}
    
    # Iterate over each residue and compute center of mass
    for residue in u.residues:
        # Get center of mass (MDAnalysis returns in Angstroms)
        com = residue.atoms.center_of_mass()
        
        # Convert from Angstroms to nm (divide by 10)
        com_nm = com / 10.0
        
        # Store with residue number (resid)
        com_coords[residue.resid] = tuple(com_nm)
    
    return com_coords


def plot_path_3d_with_bottleneck(path_row, gro_file, feature_cols=None):
    """
    Plot a 3D representation of a path showing the bottleneck region.
    
    Parameters:
    -----------
    path_row : pandas Series
        Row from representative paths DataFrame containing path info
    gro_file : str or Path
        Path to the average.gro file with molecular coordinates
    feature_cols : list
        Feature columns used for analysis
    """
    if feature_cols is None:
        feature_cols = ["E_dip", "C_dip", "rE", "Lweak_E", "pweak_E", "RE"]
    
    # Load geometry
    com_coords = read_gro_file(gro_file)
    
    # Load path information
    path_file = Path(path_row["path"])
    jce_file = path_file.with_suffix(".JCE_vectors.csv")
    
    if not jce_file.exists():
        print(f"⚠ JCE vectors file not found: {jce_file}")
        return
    
    jce_data = pd.read_csv(jce_file)
    
    # Check and normalize column names
    # Try to map to expected column names
    col_mapping = {}
    
    if "E_JxC" not in jce_data.columns:
        if "E" in jce_data.columns:
            col_mapping["E"] = "E_JxC"
        else:
            print(f"⚠ Warning: No E or E_JxC column found in {jce_file.name}")
            print(f"   Available columns: {list(jce_data.columns)}")
            return
    
    if "J_abs_mean" not in jce_data.columns:
        if "J" in jce_data.columns:
            col_mapping["J"] = "J_abs_mean"
        elif "J_abs_mean" not in jce_data.columns:
            print(f"⚠ Warning: No J or J_abs_mean column found")
    
    if "C_coherence" not in jce_data.columns:
        if "C" in jce_data.columns:
            col_mapping["C"] = "C_coherence"
        elif "C_coherence" not in jce_data.columns:
            print(f"⚠ Warning: No C or C_coherence column found")
    
    # Apply column renaming if needed
    if col_mapping:
        jce_data = jce_data.rename(columns=col_mapping)
    
    # Verify required columns exist
    required = ["site_i", "site_j", "E_JxC"]
    missing = [c for c in required if c not in jce_data.columns]
    if missing:
        print(f"⚠ Error: Missing required columns: {missing}")
        print(f"   Available columns: {list(jce_data.columns)}")
        return
    
    # Get site numbers and their coordinates
    sites = list(jce_data["site_i"]) + [jce_data["site_j"].iloc[-1]]
    
    # Extract coordinates for path sites
    path_coords = []
    valid_indices = []
    for idx, site in enumerate(sites):
        if site in com_coords:
            path_coords.append(com_coords[site])
            valid_indices.append(idx)
        else:
            print(f"Warning: Site {site} not found in .gro file")
    
    if len(path_coords) < 2:
        print("Not enough valid coordinates to plot path")
        return
    
    path_coords = np.array(path_coords)
    
    # Identify critical regions based on label
    label = path_row.get("label_primary", "Unknown")
    
    # Get E values for each dimer (use correct column names)
    E_values = jce_data["E_JxC"].values
    J_values = jce_data["J_abs_mean"].values if "J_abs_mean" in jce_data.columns else np.zeros_like(E_values)
    C_values = jce_data["C_coherence"].values if "C_coherence" in jce_data.columns else np.zeros_like(E_values)
    
    # Normalize E values for coloring (0=worst, 1=best)
    E_min, E_max = E_values.min(), E_values.max()
    if E_max > E_min:
        E_norm = (E_values - E_min) / (E_max - E_min)
    else:
        E_norm = np.ones_like(E_values)
    
    # Find bottleneck region
    bottleneck_idx = np.argmin(E_values)
    bottleneck_site_i = jce_data.loc[bottleneck_idx, "site_i"]
    bottleneck_site_j = jce_data.loc[bottleneck_idx, "site_j"]
    
    # Create figure
    fig = plt.figure(figsize=(14, 10))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot the path as connected points
    ax.plot(path_coords[:, 0], path_coords[:, 1], path_coords[:, 2], 
            'o-', color='gray', alpha=0.3, linewidth=1, markersize=4)
    
    # Color code dimers by effective transport E
    for i in range(len(jce_data)):
        if i < len(path_coords) - 1:
            # Color from blue (low E) to green (medium) to red (high E)
            color = plt.cm.RdYlGn(E_norm[i])
            ax.plot([path_coords[i, 0], path_coords[i+1, 0]],
                   [path_coords[i, 1], path_coords[i+1, 1]],
                   [path_coords[i, 2], path_coords[i+1, 2]],
                   '-', color=color, linewidth=3, alpha=0.8)
    
    # Highlight start and end
    ax.scatter(*path_coords[0], color='green', s=200, marker='o', 
              edgecolors='black', linewidths=2, label='Source', zorder=10)
    ax.scatter(*path_coords[-1], color='blue', s=200, marker='s', 
              edgecolors='black', linewidths=2, label='Target', zorder=10)
    
    # Highlight bottleneck region
    if bottleneck_idx < len(path_coords) - 1:
        ax.scatter(*path_coords[bottleneck_idx], color='red', s=300, marker='X', 
                  edgecolors='darkred', linewidths=3, label='Bottleneck', zorder=15)
        ax.scatter(*path_coords[bottleneck_idx+1], color='red', s=300, marker='X', 
                  edgecolors='darkred', linewidths=3, zorder=15)
    
    # Add labels and title
    ax.set_xlabel('X (nm)', fontsize=12)
    ax.set_ylabel('Y (nm)', fontsize=12)
    ax.set_zlabel('Z (nm)', fontsize=12)
    
    title = f"3D Path Visualization: {label}\n"
    title += f"Path: {path_file.name}"
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    # Add legend
    ax.legend(loc='upper right', fontsize=10)
    
    # Add colorbar for E values
    sm = plt.cm.ScalarMappable(cmap=plt.cm.RdYlGn, 
                               norm=plt.Normalize(vmin=E_min, vmax=E_max))
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, shrink=0.5, aspect=5)
    cbar.set_label('Effective Transport E (J×C)', rotation=270, labelpad=20, fontsize=10)
    
    # Add text box with bottleneck info
    info_text = f"BOTTLENECK (Dimer {bottleneck_idx+1}/{len(jce_data)}):\n"
    info_text += f"Sites: {bottleneck_site_i} → {bottleneck_site_j}\n"
    info_text += f"Position: {(bottleneck_idx+1)/len(jce_data):.1%} along path\n"
    
    # Add J, C, E values if available
    if "J_abs_mean" in jce_data.columns:
        info_text += f"J = {J_values[bottleneck_idx]:.4f}\n"
    if "C_coherence" in jce_data.columns:
        info_text += f"C = {C_values[bottleneck_idx]:.4f}\n"
    info_text += f"E = {E_values[bottleneck_idx]:.4f}\n"
    
    # Add reason for label based on descriptors
    info_text += f"\nLabel: {label}\n"
    if "GB-blocked" in label or "GB-crossing" in label:
        if "E_dip" in path_row.index:
            info_text += f"E_dip: {path_row['E_dip']:.3f} (mid-dip)\n"
        if "C_dip" in path_row.index:
            info_text += f"C_dip: {path_row['C_dip']:.3f} (coherence dip)\n"
        if "rE" in path_row.index:
            info_text += f"rE: {path_row['rE']:.3f} (bottleneck depth)\n"
    elif "Source-limited" in label:
        if "pC" in path_row.index or "pE" in path_row.index:
            p_val = path_row.get('pC', path_row.get('pE', 0))
            info_text += f"p: {p_val:.3f} (barrier at start)\n"
    elif "Target-limited" in label:
        if "pC" in path_row.index or "pE" in path_row.index:
            p_val = path_row.get('pC', path_row.get('pE', 1))
            info_text += f"p: {p_val:.3f} (barrier at end)\n"
    
    ax.text2D(0.02, 0.98, info_text, transform=ax.transAxes, 
             fontsize=9, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n3D visualization complete for: {label}")
    print(f"  Bottleneck at dimer {bottleneck_idx+1}/{len(jce_data)}")
    print(f"  Bottleneck E value: {E_values[bottleneck_idx]:.4f}")


# ============================================================================
# CONFIGURATION FOR 3D VISUALIZATION
# ============================================================================

ROOT_DIR = Path("./example_data")  # TODO: point this at a directory containing path_couplings_timeseries.csv files for one GB structure
# Path to the saved representative paths CSV file
REP_PATHS_FILE = ROOT_DIR / "representative_paths_for_dynamics.csv"

# Path to geometry file (.gro file with molecular coordinates)
GRO_FILE = ROOT_DIR / "averaged_ham.gro"

# Check if geometry file exists, if not provide helpful message
if not GRO_FILE.exists():
    print(f"\n⚠ WARNING: Geometry file not found: {GRO_FILE}")
    print("Please update GRO_FILE path in this cell to point to your averaged_ham.gro file")
    print("Typical locations:")
    print(f"  - {ROOT_DIR / 'averaged_ham.gro'}")
    print(f"  - {ROOT_DIR.parent / 'averaged_ham.gro'}")
else:
    print(f"\n✓ Using geometry file: {GRO_FILE}")

# ============================================================================
# RUN 3D VISUALIZATION
# ============================================================================

# Load representative paths from saved CSV file
if REP_PATHS_FILE.exists():
    print(f"\nLoading representative paths from: {REP_PATHS_FILE}")
    rep_paths = pd.read_csv(REP_PATHS_FILE)
    print(f"Found {len(rep_paths)} representative paths")
    
    # Check if geometry file exists before plotting
    if GRO_FILE.exists():
        # Plot each representative path in 3D
        for idx, row in rep_paths.iterrows():
            print(f"\n{'='*80}")
            print(f"Plotting representative path for: {row['label_primary']}")
            print(f"{'='*80}")
            plot_path_3d_with_bottleneck(row, GRO_FILE)
    else:
        print("\nSkipping 3D visualization - geometry file not found")
        print("Please update GRO_FILE path above")
else:
    print(f"\n⚠ Representative paths file not found: {REP_PATHS_FILE}")
    print("Please run the previous cells to generate representative paths first")

## Summary

This restructured notebook provides:

1. **Configuration Section**: All paths and parameters in one place
2. **Function Definitions**: All utility functions grouped together
3. **Data Processing Workflow**: Clear sequential steps
4. **Visualization Section**: All plots organized together

### Output Files Generated:

All files are saved to the configured `OUTPUT_DIR`:

| File | Description |
|------|-------------|
| `paths_symmetric_descriptors_labeled.csv` | Symmetric (order-independent) labels |
| `paths_order_descriptors_with_archetype_labels.csv` | Order-aware archetype labels |
| `paths_with_primary_labels.csv` | Primary classification (4-5 labels) |
| `primary_label_counts.csv` | Count summary for primary labels |
| `secondary_*.csv` | Secondary label compositions within primary |
| `representative_paths_for_dynamics.csv` | Representative paths (medoids) |
| `representative_paths_with_symmetric_labels.csv` | Representatives with both label types |
| `all_paths_with_both_labels.csv` | Complete dataset with dual classification |
| `primary_symmetric_crosstab_full_dataset.csv` | Cross-tabulation table |
| `primary_symmetric_percentage_full_dataset.csv` | Percentage distribution table |

### To Use This Notebook:

1. Update `ROOT_DIR` in Configuration section
2. Run cells sequentially from top to bottom
3. All outputs will be saved to `OUTPUT_DIR`
4. Visualizations are generated at the end